In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    DoubleType,
    StringType
)

paysim_schema = StructType([
    StructField("step", IntegerType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("nameOrig", StringType(), True),
    StructField("oldbalanceOrg", DoubleType(), True),
    StructField("newbalanceOrig", DoubleType(), True),
    StructField("nameDest", StringType(), True),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("isFlaggedFraud", IntegerType(), True)
])

In [0]:
from pyspark.sql.functions import current_timestamp, col

file_path = "/Volumes/paysim_fraud/bronze/raw_files/paysim.csv"

df = (
    spark.read
    .option("header", True)
    .schema(paysim_schema)
    .csv(file_path)
)

bronze_df = (
    df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
)

In [0]:
display(bronze_df.limit(10))


In [0]:
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("paysim_fraud.bronze.transactions_raw")
)

In [0]:
spark.table("paysim_fraud.bronze.transactions_raw").count()